In [1]:
"""
mv_extra_figures.py — Additional figures for Chapter 7
==========================================================
Produce quattro figure supplementari per approfondire l'analisi
of Chapter 7 of the thesis (multi-asset mean-variance).

PREREQUISITE: run mv_multiasset.py before this script,
or have data available via yfinance.

Figures produced:
  figure_ma_corr.pdf/png         — empirical correlation heatmap
  figure_ma_tikhonov.pdf/png     — sensitivity of phi_1* to parameter eta
  figure_ma_alloc_oos.pdf/png    — monthly out-of-sample allocations
  figure_ma_benchmark_compare.pdf/png — comparison of BS vs MJD benchmarks per asset

Usage:
  python mv_extra_figures.py
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib as mpl
from scipy.optimize import minimize
from scipy.stats import norm, poisson
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

mpl.rcParams.update({
    'font.family':    'serif',
    'font.size':      11,
    'axes.titlesize': 12,
    'axes.labelsize': 12,
    'legend.fontsize': 9,
})

# ─────────────────────────────────────────────────────────────────────────────
# PARAMETRI (identici a mv_multiasset.py)
# ─────────────────────────────────────────────────────────────────────────────

TICKERS     = ['^GSPC', 'AGG', 'VGK', 'GLD', 'QQQ']
NAMES       = ['S&P 500', 'Bonds', 'MSCI Europe', 'Gold', 'NASDAQ-100']
NAMES_SHORT = ['SP500', 'Bonds', 'MSCI', 'Gold', 'QQQ']
D           = len(TICKERS)
R_F         = 0.02
DT          = 1 / 12
T           = 1.0
K           = 12
Z           = 1.4
X0          = 1.0
TRAIN_END   = '2019-12-31'
TEST_END    = '2023-12-31'
ETA_DEFAULT = 0.10


# ─────────────────────────────────────────────────────────────────────────────
# CARICAMENTO DATI (stesso di mv_multiasset.py)
# ─────────────────────────────────────────────────────────────────────────────

def load_multiasset():
    import yfinance as yf
    import pandas as pd

    print("  Downloading data...")
    data = yf.download(TICKERS, start='2000-01-01', end=TEST_END, progress=False)
    if isinstance(data.columns, pd.MultiIndex):
        raw = data['Close']
    else:
        raw = data
    raw = raw.ffill().bfill()
    first_valid = raw.apply(lambda col: col.first_valid_index()).max()
    raw = raw.loc[first_valid:]

    log_daily = np.log(raw / raw.shift(1)).dropna()
    dpm = 21
    n_months = len(log_daily) // dpm
    ret_monthly = np.zeros((n_months, D))
    dates_monthly = []
    for i in range(n_months):
        block = log_daily.iloc[i*dpm:(i+1)*dpm]
        ret_monthly[i] = block.sum().values
        dates_monthly.append(pd.Timestamp(block.index[-1]))
    dates_monthly = np.array(dates_monthly)

    cutoff = pd.Timestamp(TRAIN_END)
    mask_train = np.array([d <= cutoff for d in dates_monthly])
    mask_test  = ~mask_train

    return (ret_monthly[mask_train], ret_monthly[mask_test],
            dates_monthly[mask_train], dates_monthly[mask_test])


# ─────────────────────────────────────────────────────────────────────────────
# CALIBRAZIONE (identica a mv_multiasset.py)
# ─────────────────────────────────────────────────────────────────────────────

def calibrate_bs(ret_monthly):
    mu    = ret_monthly.mean(axis=0) / DT + 0.5*(ret_monthly.std(axis=0)/np.sqrt(DT))**2
    Sigma = np.cov(ret_monthly.T) / DT
    rho   = mu - R_F
    return dict(mu=mu, Sigma=Sigma, rho=rho, SigmaJ=np.zeros((D, D)))


def calibrate_mjd(ret_monthly):
    bs = calibrate_bs(ret_monthly)
    mu_vec    = bs['mu'].copy()
    sigmaJ_sq = np.zeros(D)
    sigma_diff = np.zeros(D)

    for i in range(D):
        ri = ret_monthly[:, i]
        def neg_ll(params):
            mu_i, sig_i, lam_i, m_i, delta_i = params
            if sig_i <= 0 or lam_i <= 0 or delta_i <= 0:
                return 1e10
            kappa = np.exp(m_i + 0.5*delta_i**2) - 1
            n_max = max(10, int(lam_i*DT*5 + 20))
            ns    = np.arange(0, n_max+1)
            pn    = poisson.pmf(ns, lam_i*DT)
            pdf   = np.zeros(len(ri))
            for n, p_n in zip(ns, pn):
                if p_n < 1e-15: continue
                mu_n  = (mu_i - 0.5*sig_i**2 - lam_i*kappa)*DT + n*m_i
                var_n = sig_i**2*DT + n*delta_i**2
                pdf  += p_n * norm.pdf(ri, mu_n, np.sqrt(var_n))
            return -np.sum(np.log(np.maximum(pdf, 1e-300)))

        sig0 = np.std(ri) / np.sqrt(DT)
        x0   = [mu_vec[i], sig0, 5.0, -0.01, 0.03]
        bds  = [(None,None),(1e-4,None),(1e-4,None),(None,None),(1e-4,None)]
        try:
            res = minimize(neg_ll, x0, method='L-BFGS-B', bounds=bds,
                           options=dict(maxiter=300, ftol=1e-8))
            mu_i, sig_i, lam_i, m_i, delta_i = res.x
        except Exception:
            mu_i, sig_i = mu_vec[i], sig0
            lam_i, m_i, delta_i = 5., -0.01, 0.03

        sig_bs_i = np.std(ret_monthly[:, i]) / np.sqrt(DT)
        sigma_diff[i] = max(sig_i, 0.05 * sig_bs_i)
        sigmaJ_sq[i]  = lam_i*(np.exp(2*m_i+2*delta_i**2)
                                - 2*np.exp(m_i+0.5*delta_i**2) + 1)
        mu_vec[i] = mu_i

    corr  = np.corrcoef(ret_monthly.T)
    Sigma = np.outer(sigma_diff, sigma_diff) * corr
    try:
        sv = np.linalg.svd(Sigma, compute_uv=False)
        if sv.min() < 1e-8:
            Sigma += (1e-8 - sv.min()) * np.eye(D)
    except Exception:
        Sigma = np.diag(sigma_diff**2) + 1e-6*np.eye(D)

    SigmaJ = np.diag(sigmaJ_sq)
    rho    = mu_vec - R_F
    return dict(mu=mu_vec, Sigma=Sigma, rho=rho, SigmaJ=SigmaJ,
                sigma_diff=sigma_diff, sigmaJ_sq=sigmaJ_sq)


def phi1_star(params, eta=ETA_DEFAULT):
    Sigma_eff = params['Sigma'] + params['SigmaJ']
    eigvals   = np.linalg.eigvalsh(Sigma_eff)
    eps       = eta * np.mean(np.abs(eigvals))
    return np.linalg.solve(Sigma_eff + eps*np.eye(D), params['rho'])


# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 1 — Empirical correlation matrix heatmap
# ─────────────────────────────────────────────────────────────────────────────

def figure_correlation(ret_train):
    print("\n[Fig 1] Correlation heatmap...")
    corr = np.corrcoef(ret_train.T)

    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04,
                 label='Correlation coefficient')

    ax.set_xticks(range(D))
    ax.set_yticks(range(D))
    # escape backslash for matplotlib
    labels = ['S\\&P 500', 'Bonds', 'MSCI Europe', 'Gold', 'NASDAQ-100']
    ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=10)
    ax.set_yticklabels(labels, fontsize=10)

    for i in range(D):
        for j in range(D):
            c = corr[i, j]
            color = 'white' if abs(c) > 0.5 else 'black'
            ax.text(j, i, f'{c:.2f}', ha='center', va='center',
                    fontsize=9, color=color, fontweight='bold')

    ax.set_title('Empirical correlation matrix\\n'
                 'monthly returns 2000--2019', fontsize=11)
    plt.tight_layout()
    plt.savefig('figure_ma_corr.pdf', dpi=150, bbox_inches='tight')
    plt.savefig('figure_ma_corr.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("  Saved: figure_ma_corr.pdf/png")

    # Stampa valori notevoli
    print(f"  Correlazione SP500-NASDAQ: {corr[0,4]:.3f}")
    print(f"  Correlazione SP500-Bonds:  {corr[0,1]:.3f}")
    print(f"  Correlazione SP500-Gold:   {corr[0,3]:.3f}")


# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 2 — Sensitivity of phi_1* to Tikhonov parameter eta
# ─────────────────────────────────────────────────────────────────────────────

def figure_tikhonov_sensitivity(bs_params, mjd_params):
    print("\n[Fig 2] Tikhonov sensitivity...")

    eta_vals = np.logspace(-3, 0, 50)   # da 0.001 a 1.0
    phi1_bs_grid  = np.zeros((len(eta_vals), D))
    phi1_mjd_grid = np.zeros((len(eta_vals), D))

    for k, eta in enumerate(eta_vals):
        phi1_bs_grid[k]  = phi1_star(bs_params,  eta=eta)
        phi1_mjd_grid[k] = phi1_star(mjd_params, eta=eta)

    # Valori al default eta=0.10
    phi1_bs_def  = phi1_star(bs_params,  eta=ETA_DEFAULT)
    phi1_mjd_def = phi1_star(mjd_params, eta=ETA_DEFAULT)

    colors = ['steelblue', 'darkorange', 'forestgreen', 'firebrick', 'purple']
    labels_short = ['S\\&P 500', 'Bonds', 'MSCI Europe', 'Gold', 'NASDAQ-100']

    fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=False)

    for i in range(D):
        axes[0].semilogx(eta_vals, phi1_bs_grid[:, i],
                         color=colors[i], lw=2, label=labels_short[i])
        axes[1].semilogx(eta_vals, phi1_mjd_grid[:, i],
                         color=colors[i], lw=2, label=labels_short[i])

    for ax, phi1_def, title in zip(
            axes,
            [phi1_bs_def, phi1_mjd_def],
            ['Modello BS', 'Modello MJD']):
        ax.axvline(ETA_DEFAULT, color='black', lw=1.5, ls='--',
                   label=f'$\\eta = {ETA_DEFAULT}$ (default)')
        for i in range(D):
            ax.plot(ETA_DEFAULT, phi1_def[i], 'o',
                    color=colors[i], ms=6, zorder=5)
        ax.set_xlabel(r'Regularization level $\eta$', fontsize=11)
        ax.set_ylabel(r'$\phi_{1,i}^*$', fontsize=11)
        ax.set_title(title, fontsize=11)
        ax.legend(fontsize=8, loc='upper right')
        ax.grid(True, alpha=0.3, which='both')

    fig.suptitle(r'Sensitivity of $\boldsymbol{\phi}_1^*$ to the '
                 r'Tikhonov regularization parameter $\eta$',
                 fontsize=12)
    plt.tight_layout()
    plt.savefig('figure_ma_tikhonov.pdf', dpi=150, bbox_inches='tight')
    plt.savefig('figure_ma_tikhonov.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("  Saved: figure_ma_tikhonov.pdf/png")


# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 3 — Monthly out-of-sample allocations
# ─────────────────────────────────────────────────────────────────────────────

def figure_allocations_oos(ret_test, dates_test, phi1_learned, phi1_bs, phi1_mjd):
    print("\n[Fig 3] Out-of-sample allocations...")

    # Ricostruisce il percorso di ricchezza e le allocazioni mese per mese
    # for the three policies: RL, BS benchmark, MJD benchmark
    omega = Z  # approssimazione: omega ~ z dopo convergenza

    n_test = len(ret_test)
    wealth = {'RL': np.zeros(n_test+1),
              'BS': np.zeros(n_test+1),
              'MJD': np.zeros(n_test+1)}
    alloc  = {'RL': np.zeros((n_test, D)),
              'BS': np.zeros((n_test, D)),
              'MJD': np.zeros((n_test, D))}
    for key in wealth:
        wealth[key][0] = X0

    for k, r in enumerate(ret_test):
        t        = (k % K) * DT
        r_excess = np.exp(r) - 1 - R_F*DT

        for key, phi1 in [('RL', phi1_learned),
                           ('BS', phi1_bs),
                           ('MJD', phi1_mjd)]:
            a = -phi1 * (wealth[key][k] - omega)
            alloc[key][k] = a
            wealth[key][k+1] = wealth[key][k] + np.dot(a, r_excess)

    colors = ['steelblue', 'darkorange', 'forestgreen', 'firebrick', 'purple']
    labels_short = ['S\\&P 500', 'Bonds', 'MSCI Europe', 'Gold', 'NASDAQ-100']

    fig, axes = plt.subplots(3, 1, figsize=(13, 11), sharex=True)
    titles = ['RL Policy (learned)', 'Benchmark BS', 'Benchmark MJD']
    keys   = ['RL', 'BS', 'MJD']

    for ax, key, title in zip(axes, keys, titles):
        for i in range(D):
            ax.plot(dates_test, alloc[key][:, i],
                    color=colors[i], lw=1.5, label=labels_short[i])
        ax.axhline(0, color='black', lw=0.8, ls='--', alpha=0.5)

        # Highlight COVID crash and 2022 bear market
        import pandas as pd
        covid_start = pd.Timestamp('2020-02-01')
        covid_end   = pd.Timestamp('2020-04-30')
        bear_start  = pd.Timestamp('2022-01-01')
        bear_end    = pd.Timestamp('2022-12-31')
        ax.axvspan(covid_start, covid_end, alpha=0.12, color='red',
                   label='COVID-19 crash' if i == 0 else '')
        ax.axvspan(bear_start,  bear_end,  alpha=0.08, color='orange',
                   label='2022 bear market' if i == 0 else '')

        ax.set_ylabel(r'Allocation $a_{i,t}$', fontsize=10)
        ax.set_title(title, fontsize=11)
        ax.legend(fontsize=8, loc='upper right', ncol=3)
        ax.grid(True, alpha=0.3)

    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    axes[-1].xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
    plt.setp(axes[-1].xaxis.get_majorticklabels(), rotation=30, ha='right')
    axes[-1].set_xlabel('Date', fontsize=11)

    fig.suptitle('Monthly out-of-sample allocations 2020--2023',
                 fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig('figure_ma_alloc_oos.pdf', dpi=150, bbox_inches='tight')
    plt.savefig('figure_ma_alloc_oos.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("  Saved: figure_ma_alloc_oos.pdf/png")


# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 4 — Comparison of phi_1* BS vs MJD per asset
# ─────────────────────────────────────────────────────────────────────────────

def figure_benchmark_compare(bs_params, mjd_params, phi1_learned):
    print("\n[Fig 4] Benchmark comparison BS vs MJD...")

    phi1_bs  = phi1_star(bs_params)
    phi1_mjd = phi1_star(mjd_params)

    x = np.arange(D)
    width = 0.25

    fig, ax = plt.subplots(figsize=(9, 5))

    bars1 = ax.bar(x - width, phi1_bs,      width, color='steelblue',
                   alpha=0.85, label=r'$\boldsymbol{\phi}_1^*$ BS')
    bars2 = ax.bar(x,          phi1_mjd,     width, color='darkorange',
                   alpha=0.85, label=r'$\boldsymbol{\phi}_1^*$ MJD')
    bars3 = ax.bar(x + width,  phi1_learned, width, color='forestgreen',
                   alpha=0.85, label=r'$\boldsymbol{\phi}_1$ learned (RL)')

    # Annotazioni valori
    for bars in [bars1, bars2, bars3]:
        for bar in bars:
            h = bar.get_height()
            ax.annotate(f'{h:.2f}',
                        xy=(bar.get_x() + bar.get_width()/2, h),
                        xytext=(0, 3), textcoords='offset points',
                        ha='center', va='bottom', fontsize=8)

    ax.set_xticks(x)
    labels_short = ['S\\&P 500', 'Bonds', 'MSCI Europe', 'Gold', 'NASDAQ-100']
    ax.set_xticklabels(labels_short, fontsize=10)
    ax.set_ylabel(r'Value of $\phi_{1,i}$', fontsize=11)
    ax.set_title(r'Comparison of $\boldsymbol{\phi}_1^*$ (BS, MJD) and '
                 r'learned $\boldsymbol{\phi}_1$ per asset',
                 fontsize=11)
    ax.legend(fontsize=10)
    ax.axhline(0, color='black', lw=0.8)
    ax.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('figure_ma_benchmark_compare.pdf', dpi=150, bbox_inches='tight')
    plt.savefig('figure_ma_benchmark_compare.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("  Saved: figure_ma_benchmark_compare.pdf/png")

    # Stampa differenze percentuali MJD vs BS
    print("\n  Percentage difference MJD vs BS per asset:")
    for i, name in enumerate(NAMES):
        diff = (phi1_mjd[i] - phi1_bs[i]) / abs(phi1_bs[i]) * 100
        print(f"    {name:<20}: {diff:+.1f}%")


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == '__main__':

    print("\n[0] Loading data...")
    ret_train, ret_test, dates_train, dates_test = load_multiasset()

    print("\n[0] Calibrating parameters (training set 2000-2019)...")
    bs_params  = calibrate_bs(ret_train)
    mjd_params = calibrate_mjd(ret_train)

    phi1_bs      = phi1_star(bs_params)
    phi1_mjd     = phi1_star(mjd_params)
    # Learned value from main run of mv_multiasset.py
    phi1_learned = np.array([0.800, 1.603, 1.177, 0.657, 0.588])

    print(f"\n  phi_1* BS      = {phi1_bs}")
    print(f"  phi_1* MJD     = {phi1_mjd}")
    print(f"  phi_1  learned = {phi1_learned}")

    figure_correlation(ret_train)
    figure_tikhonov_sensitivity(bs_params, mjd_params)
    figure_allocations_oos(ret_test, dates_test, phi1_learned, phi1_bs, phi1_mjd)
    figure_benchmark_compare(bs_params, mjd_params, phi1_learned)

    print("\n" + "="*55)
    print("  All figures produced:")
    print("    figure_ma_corr.pdf/png")
    print("    figure_ma_tikhonov.pdf/png")
    print("    figure_ma_alloc_oos.pdf/png")
    print("    figure_ma_benchmark_compare.pdf/png")
    print("="*55)


[0] Loading data...

[0] Calibrating parameters (training set 2000-2019)...

  phi_1* BS      = [2.80522271 1.84526337 0.69865185 0.27088384 0.2921808 ]
  phi_1* MJD     = [ 3.09160467  2.27088415  0.76363904 -0.60050432  0.93092292]
  phi_1  learned = [0.8   1.603 1.177 0.657 0.588]

[Fig 1] Correlation heatmap...
  Saved: figure_ma_corr.pdf/png
  Correlazione SP500-NASDAQ: 0.073
  Correlazione SP500-Bonds:  0.007
  Correlazione SP500-Gold:   0.052

[Fig 2] Tikhonov sensitivity...
  Saved: figure_ma_tikhonov.pdf/png

[Fig 3] Out-of-sample allocations...
  Saved: figure_ma_alloc_oos.pdf/png

[Fig 4] Benchmark comparison BS vs MJD...
  Saved: figure_ma_benchmark_compare.pdf/png

  Percentage difference MJD vs BS per asset:
    S&P 500             : +10.2%
    Bonds               : +23.1%
    MSCI Europe         : +9.3%
    Gold                : -321.7%
    NASDAQ-100          : +218.6%

  All figures produced:
    figure_ma_corr.pdf/png
    figure_ma_tikhonov.pdf/png
    figure_ma_allo